<a href="https://colab.research.google.com/github/alifihsan7/kelompok9-image-captioning/blob/main/notebooks/02_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

# 1. Menetapkan asumsi jalur folder proyek di Google Drive
# Menangani variasi penamaan default MyDrive / My Drive di Colab
CANDIDATE_PATHS = [
    Path("/content/drive/MyDrive/Kelompok9"),
    Path("/content/drive/My Drive/Kelompok9"),
]

project_dir = None
for path in CANDIDATE_PATHS:
    if path.is_dir():
        project_dir = path
        break

# 2 & 3. Pemeriksaan keberadaan folder utama dan subfolder src
if project_dir is None:
    # 5. Pesan jika folder utama tidak ditemukan
    print("❌ ERROR: Folder utama 'Kelompok9' tidak ditemukan di Google Drive.")
    print("Pemeriksaan yang perlu dilakukan:")
    print("1. Pastikan mount Google Drive berhasil (`drive.mount('/content/drive')`).")
    print("2. Jika 'Kelompok9' adalah folder bersama (Shared Folder), pastikan sudah menambahkan pintasan ke Drive ('Add shortcut to Drive') pada akun Anda.")
    print("3. Pastikan penulisan nama folder tepat 'Kelompok9' (sensitif huruf besar/kecil).")
else:
    # 2. Menentukan lokasi subfolder src
    src_dir = project_dir / "src"

    # 4. Menampilkan lokasi yang ditemukan
    print(f"✅ Folder utama proyek ditemukan : {project_dir}")

    # 3 & 5. Pemeriksaan ketersediaan subfolder src
    if src_dir.is_dir():
        print(f"✅ Subfolder src ditemukan        : {src_dir}")
        print("\nStatus: Jalur valid. Anda siap menyimpan 'src/model.py'.")
    else:
        print(f"❌ ERROR: Subfolder 'src' tidak ditemukan di dalam {project_dir}.")
        print("Peringatan: Folder baru tidak dibuat secara otomatis.")
        print(f"Silakan buat folder 'src' di Google Drive secara manual pada lokasi: {project_dir}")

✅ Folder utama proyek ditemukan : /content/drive/MyDrive/Kelompok9
✅ Subfolder src ditemukan        : /content/drive/MyDrive/Kelompok9/src

Status: Jalur valid. Anda siap menyimpan 'src/model.py'.


Mengatur Direktori Kerja dan Menyimpan src/model.py

In [3]:
import os
import sys

# Pindah ke direktori utama proyek di Drive yang sudah terverifikasi sebelumnya
os.chdir(project_dir)

# Tambahkan direktori proyek ke sys.path agar impor modul 'src' berjalan lancar
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print(f"Direktori kerja aktif saat ini: {os.getcwd()}")

Direktori kerja aktif saat ini: /content/drive/MyDrive/Kelompok9


In [4]:
%%writefile /content/drive/MyDrive/Kelompok9/src/model.py
"""
src/model.py
Modul arsitektur Decoder LSTM untuk Image Captioning.
Mengikuti spesifikasi Kontrak Teknis Data V1 (Kelompok 9).
"""

from typing import List, Dict, Any
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CaptionDecoder(nn.Module):
    """
    Decoder LSTM untuk Image Captioning.

    Arsitektur:
        - Proyeksi linear fitur visual [B, feature_dim] ke status awal LSTM (h0, c0).
        - Lapisan embedding token kata dengan penanganan padding_idx.
        - LSTM 1 lapis (batch_first=True) untuk pemrosesan sekuens teks.
        - Regulerisasi Dropout pada representasi status tersembunyi.
        - Lapisan linear proyeksi akhir ke dimensi kosakata (raw logits).

    Args:
        vocab_size (int): Jumlah total kosakata unik (V).
        feature_dim (int, optional): Dimensi vektor fitur visual encoder ResNet-50. Default: 2048.
        embed_dim (int, optional): Dimensi representasi vektor kata. Default: 256.
        hidden_size (int, optional): Dimensi ruang status tersembunyi (hidden & cell state). Default: 512.
        dropout_p (float, optional): Probabilitas dropout regulerisasi. Default: 0.5.
        pad_idx (int, optional): Indeks token khusus <pad>. Default: 0.
    """

    def __init__(
        self,
        vocab_size: int,
        feature_dim: int = 2048,
        embed_dim: int = 256,
        hidden_size: int = 512,
        dropout_p: float = 0.5,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.feature_dim = feature_dim
        self.embed_dim = embed_dim
        self.hidden_size = hidden_size
        self.dropout_p = dropout_p
        self.pad_idx = pad_idx

        # 1. Proyeksi fitur visual ke hidden state (h0) dan cell state (c0) awal
        # [B, feature_dim] -> [B, hidden_size]
        self.init_h = nn.Linear(feature_dim, hidden_size)
        self.init_c = nn.Linear(feature_dim, hidden_size)

        # 2. Lapisan penyematan kata (word embedding)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx,
        )

        # 3. Lapisan recurrent LSTM 1 lapis
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        # 4. Lapisan regulerisasi dropout
        self.dropout = nn.Dropout(p=dropout_p)

        # 5. Lapisan linear keluaran menuju ruang kosakata
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(
        self, features: torch.Tensor, captions_in: torch.Tensor
    ) -> torch.Tensor:
        """
        Menjalankan satu lintasan komputasi maju menggunakan skema Teacher Forcing.

        Bentuk Tensor:
            - features:    torch.FloatTensor [B, feature_dim]
            - captions_in: torch.LongTensor  [B, L]
            - logits:      torch.FloatTensor [B, L, vocab_size]

        Args:
            features (torch.Tensor): Representasi visual dari encoder berdimensi [B, feature_dim].
            captions_in (torch.Tensor): Sekuens token masukan berdimensi [B, L].

        Returns:
            torch.Tensor: Logits (skor mentah sebelum aktivasi softmax) berdimensi [B, L, vocab_size].
        """
        # 1. Inisialisasi h0 dan c0 dari fitur visual
        # [B, feature_dim] -> [B, hidden_size] -> [1, B, hidden_size]
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)

        # 2. Pemetaan token teks ke ruang kontinu
        # [B, L] -> [B, L, embed_dim]
        embedded = self.embedding(captions_in)

        # 3. Propagasi sekuensial LSTM
        # lstm_out berdimensi [B, L, hidden_size]
        lstm_out, _ = self.lstm(embedded, (h0, c0))

        # 4. Penerapan dropout pada luaran hidden state
        out = self.dropout(lstm_out)

        # 5. Proyeksi skor mentah ke kosakata tanpa softmax
        # [B, L, hidden_size] -> [B, L, vocab_size]
        logits = self.fc_out(out)

        return logits

    def generate_greedy(
        self,
        feature: torch.Tensor,
        max_len: int = 30,
        start_token_id: int = 1,
        end_token_id: int = 2,
        pad_token_id: int = 0,
    ) -> List[int]:
        """
        Membangkitkan token deskripsi gambar secara greedy langkah demi langkah (autoregresif).

        Perbedaan Batas Panjang:
            - max_len=30 di sini menentukan batas maksimal langkah inferensi (maksimal 30 token prediksi).
            - Ini berbeda dari panjang sekuens training T=30 yang mencakup <start>, kata isi, <end>, dan <pad>.

        Bentuk Tensor Masukan:
            - feature: [feature_dim] (1D) atau [1, feature_dim] (2D)

        Args:
            feature (torch.Tensor): Fitur visual citra tunggal berdimensi [feature_dim] atau [1, feature_dim].
            max_len (int, optional): Batas maksimal langkah prediksi kata. Default: 30.
            start_token_id (int, optional): ID token khusus <start>. Default: 1.
            end_token_id (int, optional): ID token khusus <end>. Default: 2.
            pad_token_id (int, optional): ID token khusus <pad>. Default: 0.

        Returns:
            List[int]: Daftar indeks kata hasil inferensi (tanpa token <start>, <end>, maupun <pad>).

        Raises:
            ValueError: Jika bentuk tensor fitur tidak sesuai dengan dimensi [feature_dim] atau [1, feature_dim].
        """
        if feature.dim() == 1:
            if feature.shape[0] != self.feature_dim:
                raise ValueError(
                    f"Fitur gambar 1D harus berukuran [{self.feature_dim}], "
                    f"tetapi diterima bentuk {list(feature.shape)}."
                )
            feature = feature.unsqueeze(0)
        elif feature.dim() == 2:
            if feature.shape != (1, self.feature_dim):
                raise ValueError(
                    f"Fitur gambar 2D harus berukuran [1, {self.feature_dim}], "
                    f"tetapi diterima bentuk {list(feature.shape)}."
                )
        else:
            raise ValueError(
                f"Fitur gambar harus berdimensi 1D [{self.feature_dim}] atau 2D [1, {self.feature_dim}], "
                f"tetapi diterima tensor {feature.dim()}D dengan bentuk {list(feature.shape)}."
            )

        target_device = next(self.parameters()).device
        if feature.device != target_device:
            feature = feature.to(target_device)

        was_training = self.training
        self.eval()

        generated_tokens: List[int] = []

        try:
            with torch.no_grad():
                h = self.init_h(feature).unsqueeze(0)
                c = self.init_c(feature).unsqueeze(0)

                current_token = torch.tensor(
                    [[start_token_id]], dtype=torch.long, device=target_device
                )

                for _ in range(max_len):
                    emb = self.embedding(current_token)
                    lstm_out, (h, c) = self.lstm(emb, (h, c))
                    logits = self.fc_out(lstm_out.squeeze(1))

                    logits[0, pad_token_id] = -float("inf")
                    logits[0, start_token_id] = -float("inf")

                    pred_token = torch.argmax(logits, dim=-1).item()

                    if pred_token == end_token_id:
                        break

                    generated_tokens.append(pred_token)
                    current_token = torch.tensor(
                        [[pred_token]], dtype=torch.long, device=target_device
                    )
        finally:
            self.train(was_training)

        return generated_tokens

    def generate_beam(
        self,
        feature: torch.Tensor,
        beam_size: int = 3,
        max_len: int = 30,
        alpha: float = 0.7,
        start_token_id: int = 1,
        end_token_id: int = 2,
        pad_token_id: int = 0,
    ) -> List[int]:
        """
        Membangkitkan deskripsi gambar menggunakan pencarian beam (Beam Search).

        Mekanisme:
            - Memelihara satu kumpulan beam dengan kapasitas maksimal beam_size kandidat (aktif dan selesai).
            - Mengakumulasi log-probabilitas via log_softmax pada setiap langkah waktu.
            - Status LSTM (h, c) memiliki definisi waktu konsisten: status awal h0/c0 diproyeksikan dari gambar,
              seluruh cabang anak dari induk yang sama berbagi h/c hasil langkah induk, dan status baru bercabang
              setelah token masing-masing diproses tanpa modifikasi in-place.
            - Token PAD (0) dan START (1) dimask ke -inf setelah log_softmax, dan skor non-finite dibuang.
            - Pemilihan top-k ekspansi menggunakan stable argsort sehingga jika skor identik, token dengan ID
              lebih kecil diprioritaskan sebelum pemotongan k_safe.
            - Kandidat selesai ditandai saat memprediksi token END (2). Token END ikut dihitung ke dalam skor
              dan panjang L, tetapi tidak dimasukkan ke dalam daftar kembalian tokens.
            - Normalisasi panjang tunggal: score = cumulative_log_prob / (((5 + L) / 6) ** alpha).
            - Pemilihan akhir diurutkan secara deterministik berdasarkan skor normalisasi tertinggi.

        Args:
            feature (torch.Tensor): Fitur visual citra tunggal berdimensi [feature_dim] atau [1, feature_dim].
            beam_size (int, optional): Jumlah kandidat paralel yang dipertahankan. Default: 3.
            max_len (int, optional): Batas maksimal langkah iterasi pembangkitan kata. Default: 30.
            alpha (float, optional): Eksponen penalti panjang sekuens. Default: 0.7.
            start_token_id (int, optional): ID token khusus <start>. Default: 1.
            end_token_id (int, optional): ID token khusus <end>. Default: 2.
            pad_token_id (int, optional): ID token khusus <pad>. Default: 0.

        Returns:
            List[int]: Daftar indeks kata hasil inferensi (tanpa token <start>, <end>, maupun <pad>).

        Raises:
            ValueError: Jika bentuk feature tidak valid atau beam_size < 1.
            RuntimeError: Jika tidak ada kandidat valid yang tersisa dalam beam.
        """
        # 1. Validasi bentuk tensor input
        if feature.dim() == 1:
            if feature.shape[0] != self.feature_dim:
                raise ValueError(
                    f"Fitur gambar 1D harus berukuran [{self.feature_dim}], "
                    f"tetapi diterima bentuk {list(feature.shape)}."
                )
            feature = feature.unsqueeze(0)
        elif feature.dim() == 2:
            if feature.shape != (1, self.feature_dim):
                raise ValueError(
                    f"Fitur gambar 2D harus berukuran [1, {self.feature_dim}], "
                    f"tetapi diterima bentuk {list(feature.shape)}."
                )
        else:
            raise ValueError(
                f"Fitur gambar harus berdimensi 1D [{self.feature_dim}] atau 2D [1, {self.feature_dim}], "
                f"tetapi diterima tensor {feature.dim()}D dengan bentuk {list(feature.shape)}."
            )

        if beam_size < 1:
            raise ValueError(f"beam_size harus >= 1, tetapi diterima {beam_size}.")

        # 2. Penyelarasan device antara bobot model dan tensor operasional
        target_device = next(self.parameters()).device
        if feature.device != target_device:
            feature = feature.to(target_device)

        # 3. Manajemen mode model via try/finally
        was_training = self.training
        self.eval()

        try:
            with torch.no_grad():
                h0 = self.init_h(feature).unsqueeze(0)
                c0 = self.init_c(feature).unsqueeze(0)

                # Kandidat awal: START diproses pada langkah pertama
                # START tidak dihitung dalam L (L = 0)
                initial_candidate: Dict[str, Any] = {
                    "tokens": [],
                    "cum_log_prob": 0.0,
                    "score": 0.0,
                    "h": h0,
                    "c": c0,
                    "is_done": False,
                    "length": 0,
                }
                beam: List[Dict[str, Any]] = [initial_candidate]

                for _ in range(max_len):
                    all_candidates: List[Dict[str, Any]] = []

                    for cand in beam:
                        if cand["is_done"]:
                            # Pertahankan kandidat selesai tanpa diekspansi lagi dan tanpa mengubah skor
                            all_candidates.append(cand)
                            continue

                        # Tentukan input token untuk langkah aktif
                        if len(cand["tokens"]) == 0:
                            input_token_id = start_token_id
                        else:
                            input_token_id = cand["tokens"][-1]

                        current_token = torch.tensor(
                            [[input_token_id]], dtype=torch.long, device=target_device
                        )

                        emb = self.embedding(current_token)
                        lstm_out, (next_h, next_c) = self.lstm(emb, (cand["h"], cand["c"]))
                        logits = self.fc_out(lstm_out.squeeze(1))

                        # Log-softmax untuk akumulasi probabilitas numerik stabil
                        log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

                        # Masking PAD dan START menjadi -inf
                        log_probs[pad_token_id] = -float("inf")
                        log_probs[start_token_id] = -float("inf")

                        # Ambil hanya token dengan nilai finite (membuang PAD, START, dan skor non-finite)
                        finite_mask = torch.isfinite(log_probs)
                        finite_indices = torch.nonzero(finite_mask, as_tuple=False).squeeze(-1)

                        if finite_indices.numel() > 0:
                            finite_log_probs = log_probs[finite_indices]
                            # Pengurutan stabil descending: jika skor sama, token ID lebih kecil diprioritaskan
                            sort_perm = torch.argsort(finite_log_probs, descending=True, stable=True)
                            k_safe = min(beam_size, sort_perm.numel())
                            selected_indices = finite_indices[sort_perm[:k_safe]].tolist()
                            selected_vals = finite_log_probs[sort_perm[:k_safe]].tolist()

                            for val, idx in zip(selected_vals, selected_indices):
                                new_cum = cand["cum_log_prob"] + val
                                new_len = cand["length"] + 1

                                # Rumus normalisasi panjang tunggal
                                length_penalty = ((5.0 + new_len) / 6.0) ** alpha
                                new_score = new_cum / length_penalty

                                if idx == end_token_id:
                                    # END dihitung dalam skor dan panjang L, tetapi dibuang dari list tokens
                                    all_candidates.append({
                                        "tokens": list(cand["tokens"]),
                                        "cum_log_prob": new_cum,
                                        "score": new_score,
                                        "h": next_h,
                                        "c": next_c,
                                        "is_done": True,
                                        "length": new_len,
                                    })
                                else:
                                    all_candidates.append({
                                        "tokens": cand["tokens"] + [idx],
                                        "cum_log_prob": new_cum,
                                        "score": new_score,
                                        "h": next_h,
                                        "c": next_c,
                                        "is_done": False,
                                        "length": new_len,
                                    })

                    # Validasi ketersediaan kandidat valid
                    if len(all_candidates) == 0:
                        raise RuntimeError("Tidak ada kandidat valid yang tersisa dalam beam search.")

                    # Seleksi global K terbaik dengan tie-breaker deterministik
                    # (-score, tokens, -is_done)
                    all_candidates.sort(key=lambda x: (-x["score"], x["tokens"], -int(x["is_done"])))
                    beam = all_candidates[:beam_size]

                    # Hentikan jika seluruh kandidat yang benar-benar tersimpan telah selesai
                    if len(beam) > 0 and all(c["is_done"] for c in beam):
                        break

                if len(beam) == 0:
                    raise RuntimeError("Tidak ada kandidat valid yang tersisa dalam beam search.")

                # Pilih kandidat dengan skor ternormalisasi tertinggi di antara semua kandidat tersisa
                beam.sort(key=lambda x: (-x["score"], x["tokens"], -int(x["is_done"])))
                best_candidate = beam[0]
                return best_candidate["tokens"]
        finally:
            self.train(was_training)

Overwriting /content/drive/MyDrive/Kelompok9/src/model.py


In [5]:
import importlib
import torch
import torch.nn as nn

# Reload modul agar perubahan generate_greedy termuat
import src.model
importlib.reload(src.model)
from src.model import CaptionDecoder

print("--- MEMULAI PENGUJIAN UNIT GENERATE_GREEDY ---")

# 1. Inisialisasi Model Uji
torch.manual_seed(42)
vocab_size = 3000
feature_dim = 2048
model = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim)

# Menentukan device aktif model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# -------------------------------------------------------------------------
# Uji 1: Validasi Bentuk Input 1D dan 2D serta Penolakan Bentuk Salah
# -------------------------------------------------------------------------
f_1d = torch.randn(feature_dim, device=device)
f_2d = torch.randn(1, feature_dim, device=device)

tokens_1d = model.generate_greedy(f_1d, max_len=10)
tokens_2d = model.generate_greedy(f_2d, max_len=10)

assert isinstance(tokens_1d, list), "Keluaran input 1D harus berupa list"
assert isinstance(tokens_2d, list), "Keluaran input 2D harus berupa list"
print("✅ Uji 1a: Input 1D [2048] dan 2D [1, 2048] berhasil diproses.")

# Uji penolakan bentuk input invalid
shape_error_caught = 0
for bad_shape in [(1024,), (32, 2048), (1, 1, 2048)]:
    try:
        bad_tensor = torch.randn(*bad_shape, device=device)
        model.generate_greedy(bad_tensor)
    except ValueError as e:
        shape_error_caught += 1

assert shape_error_caught == 3, "Model harus menolak bentuk selain [2048] dan [1, 2048]"
print("✅ Uji 1b: Penanganan ValueError untuk input tidak valid berfungsi dengan benar.")

# -------------------------------------------------------------------------
# Uji 2: Pengecekan Masking PAD (0), START (1), dan Pengecualian END (2)
# -------------------------------------------------------------------------
# Hasil token keluaran tidak boleh mengandung token 0, 1, maupun 2
for token_list in [tokens_1d, tokens_2d]:
    assert 0 not in token_list, "Error: Token <pad> (0) ditemukan pada hasil!"
    assert 1 not in token_list, "Error: Token <start> (1) ditemukan pada hasil!"
    assert 2 not in token_list, "Error: Token <end> (2) ditemukan pada hasil!"
    assert all(isinstance(t, int) for t in token_list), "Seluruh elemen harus bertipe int."
print("✅ Uji 2: Output list[int] bersih tanpa token PAD (0), START (1), maupun END (2).")

# -------------------------------------------------------------------------
# Uji 3: Batas Panjang & Validasi Masking PAD/START (Kasus Terkontrol)
# -------------------------------------------------------------------------
# 1. Inisialisasi model uji terpisah
model_controlled_3 = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)

with torch.no_grad():
    # 2. Nolkan bobot fc_out agar skor murni ditentukan oleh bias (independen dari representasi LSTM)
    model_controlled_3.fc_out.weight.zero_()

    # 3. Konfigurasi bias terkontrol
    # Set seluruh token (termasuk END=2 dan token lainnya) ke nilai rendah
    model_controlled_3.fc_out.bias.fill_(-10.0)

    # PAD (0) dan START (1) diberi bias tertinggi
    model_controlled_3.fc_out.bias[0] = 100.0  # PAD (<pad>=0)
    model_controlled_3.fc_out.bias[1] = 90.0   # START (<start>=1)

    # Token isi target (ID 4) menjadi kandidat tertinggi berikutnya
    model_controlled_3.fc_out.bias[4] = 50.0

# 4. Jalankan inferensi dengan batas langkah max_len=15
max_test_len = 15
tokens_uji3 = model_controlled_3.generate_greedy(f_2d, max_len=max_test_len)

# 5. Validasi:
# - Jika masking gagal, argmax akan memilih token 0 (<pad>) atau token 1 (<start>)
# - Karena di-mask ke -inf, argmax memilih token 4 pada setiap langkah
# - Karena END (2) tidak terpilih, inferensi berhenti tepat pada max_len langkah
expected_tokens = [4] * max_test_len

assert tokens_uji3 == expected_tokens, (
    f"Keluaran tidak sesuai! Diharapkan {expected_tokens}, tetapi diperoleh {tokens_uji3}"
)
assert len(tokens_uji3) == max_test_len, (
    f"Panjang keluaran salah! Diharapkan {max_test_len}, didapat {len(tokens_uji3)}"
)

print(f"✅ Uji 3: Kasus terkontrol lolos — masking PAD/START terbukti aktif dan loop berhenti tepat pada max_len={max_test_len} ({tokens_uji3[:3]}...).")

# -------------------------------------------------------------------------
# Uji 4: Berhenti Ketika Token END Terpilih (Kasus Terkontrol)
# -------------------------------------------------------------------------
# Membuat model kontrol di mana token <end> (indeks 2) dipaksa memiliki bobot/bias tertinggi
model_controlled = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)

with torch.no_grad():
    # Set bias output sangat negatif, kecuali untuk token <end>
    model_controlled.fc_out.bias.fill_(-1000.0)
    model_controlled.fc_out.bias[2] = 1000.0  # Token END dipastikan terpilih pada langkah pertama

tokens_end_immediate = model_controlled.generate_greedy(f_2d, max_len=30)
assert len(tokens_end_immediate) == 0, f"Seharusnya langsung berhenti tanpa menambah token, didapat len={len(tokens_end_immediate)}"
assert 2 not in tokens_end_immediate, "Token END tidak boleh dimasukkan ke dalam list"
print("✅ Uji 4: Kasus terkontrol berhasil berhenti seketika saat token <end> terpilih.")

# -------------------------------------------------------------------------
# Uji 5: Pemulihan Status Mode Pelatihan (train vs eval)
# -------------------------------------------------------------------------
# Kasus 5a: Mode awal train -> harus kembali train
model.train()
assert model.training is True
_ = model.generate_greedy(f_2d, max_len=5)
assert model.training is True, "Gagal memulihkan model ke mode train!"

# Kasus 5b: Mode awal eval -> harus tetap eval
model.eval()
assert model.training is False
_ = model.generate_greedy(f_2d, max_len=5)
assert model.training is False, "Gagal mempertahankan model dalam mode eval!"
print("✅ Uji 5: Pemulihan status model (train maupun eval) via try/finally terverifikasi aman.")

# -------------------------------------------------------------------------
# Uji 6: Integritas Gradien (Tidak Membuat Maupun Mengubah Gradien)
# -------------------------------------------------------------------------
# Siapkan gradien pada model terlebih dahulu (mensimulasikan backward pass sebelumnya)
model.train()
dummy_feat_train = torch.randn(2, feature_dim, device=device)
dummy_captions_train = torch.randint(0, vocab_size, (2, 29), device=device)
dummy_logits = model(dummy_feat_train, dummy_captions_train)
dummy_loss = dummy_logits.sum()
dummy_loss.backward()

# Ambil salinan persis nilai gradien yang ada
grad_h_before = model.init_h.weight.grad.clone()
assert grad_h_before is not None, "Gradien awal model harus ada."

# Jalankan inferensi greedy menggunakan tensor masukan yang memiliki requires_grad=True
f_grad_test = torch.randn(feature_dim, device=device, requires_grad=True)
_ = model.generate_greedy(f_grad_test, max_len=5)

# Periksa bahwa tensor input tidak menerima gradien baru
assert f_grad_test.grad is None, "Input tensor inferensi tidak boleh menghasilkan gradien!"

# Periksa bahwa gradien bobot model yang sudah ada TIDAK berubah sedikit pun
assert torch.equal(model.init_h.weight.grad, grad_h_before), "Gradien model yang sudah ada terubah selama inferensi!"
print("✅ Uji 6: Inferensi tidak membuat gradien baru dan tidak mengubah gradien yang sudah ada.")

print("\n--- SEMUA PENGUJIAN UNIT GENERATE_GREEDY BERHASIL DILALUI ---")


--- MEMULAI PENGUJIAN UNIT GENERATE_GREEDY ---
✅ Uji 1a: Input 1D [2048] dan 2D [1, 2048] berhasil diproses.
✅ Uji 1b: Penanganan ValueError untuk input tidak valid berfungsi dengan benar.
✅ Uji 2: Output list[int] bersih tanpa token PAD (0), START (1), maupun END (2).
✅ Uji 3: Kasus terkontrol lolos — masking PAD/START terbukti aktif dan loop berhenti tepat pada max_len=15 ([4, 4, 4]...).
✅ Uji 4: Kasus terkontrol berhasil berhenti seketika saat token <end> terpilih.
✅ Uji 5: Pemulihan status model (train maupun eval) via try/finally terverifikasi aman.
✅ Uji 6: Inferensi tidak membuat gradien baru dan tidak mengubah gradien yang sudah ada.

--- SEMUA PENGUJIAN UNIT GENERATE_GREEDY BERHASIL DILALUI ---


Verifikasi Fisik File dan Uji Impor Modul

In [6]:
from pathlib import Path
import importlib

target_file = project_dir / "src" / "model.py"

if target_file.is_file():
    print(f"✅ Berkas berhasil diverifikasi di: {target_file}")

    # Impor kelas CaptionDecoder dari berkas src/model.py
    import src.model
    importlib.reload(src.model)  # Memastikan versi terbaru yang dimuat jika ada perubahan
    from src.model import CaptionDecoder

    print("✅ Berhasil mengimpor 'CaptionDecoder' dari 'src.model'.")
else:
    print(f"❌ ERROR: Berkas {target_file} belum ditemukan di Google Drive.")

✅ Berkas berhasil diverifikasi di: /content/drive/MyDrive/Kelompok9/src/model.py
✅ Berhasil mengimpor 'CaptionDecoder' dari 'src.model'.


Uji Fungsional Tensor Dummy, Loss, dan Aliran Gradien

In [7]:
import torch
import torch.nn as nn

# 1. Parameter tensor pengujian
B = 32
feature_dim = 2048
L = 29
vocab_size = 3000

# 2. Penyiapan tensor dummy
torch.manual_seed(42)
dummy_features = torch.randn(B, feature_dim, dtype=torch.float32)
dummy_captions_in = torch.randint(0, vocab_size, (B, L), dtype=torch.long)
dummy_targets = torch.randint(0, vocab_size, (B, L), dtype=torch.long)

# 3. Inisialisasi model
model = CaptionDecoder(
    vocab_size=vocab_size,
    feature_dim=feature_dim,
    embed_dim=256,
    hidden_size=512,
    dropout_p=0.5,
    pad_idx=0
)

# 4. Uji forward pass
logits = model(dummy_features, dummy_captions_in)
print(f"Bentuk tensor logits   : {list(logits.shape)}")
assert logits.shape == (B, L, vocab_size), f"Dimensi logits tidak cocok! Diharapkan {(B, L, vocab_size)}, didapat {logits.shape}"
print("✅ Bentuk logits sesuai kontrak [B, L, vocab_size].")

# 5. Uji kalkulasi loss (perataan sesuai kontrak: [B*L, V] dan [B*L])
criterion = nn.CrossEntropyLoss(ignore_index=0)
loss = criterion(logits.reshape(-1, vocab_size), dummy_targets.reshape(-1))
print(f"Nilai loss dummy       : {loss.item():.4f}")

# 6. Uji backward pass dan verifikasi gradien
model.zero_grad()
loss.backward()

grad_h = model.init_h.weight.grad
grad_c = model.init_c.weight.grad
grad_fc = model.fc_out.weight.grad

assert grad_h is not None and grad_h.norm().item() > 0, "Gradien pada init_h tidak mengalir!"
assert grad_c is not None and grad_c.norm().item() > 0, "Gradien pada init_c tidak mengalir!"
assert grad_fc is not None and grad_fc.norm().item() > 0, "Gradien pada fc_out tidak mengalir!"

print(f"Norm gradien init_h    : {grad_h.norm().item():.6f} (mengalir)")
print(f"Norm gradien init_c    : {grad_c.norm().item():.6f} (mengalir)")
print(f"Norm gradien fc_out    : {grad_fc.norm().item():.6f} (mengalir)")
print("\n✅ Seluruh pengujian forward pass, perataan loss, dan aliran gradien BERHASIL.")

Bentuk tensor logits   : [32, 29, 3000]
✅ Bentuk logits sesuai kontrak [B, L, vocab_size].
Nilai loss dummy       : 8.0069
Norm gradien init_h    : 0.034732 (mengalir)
Norm gradien init_c    : 0.064243 (mengalir)
Norm gradien fc_out    : 0.116075 (mengalir)

✅ Seluruh pengujian forward pass, perataan loss, dan aliran gradien BERHASIL.


PENGUJIAN UNIT GENERATE_BEAM

In [8]:
import importlib
import torch
import torch.nn as nn

# Reload modul model agar perubahan termutakhir dimuat
import src.model
importlib.reload(src.model)
from src.model import CaptionDecoder

print("--- MEMULAI PENGUJIAN UNIT GENERATE_BEAM ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = 3000
feature_dim = 2048

# -------------------------------------------------------------------------
# Uji 1: Skenario Skor Seri (Tie-Breaker) Dua Token Isi Tertinggi
# Membuktikan stable sort memilih token ID terkecil dan hasilnya identik dengan argmax greedy
# -------------------------------------------------------------------------
model_tie = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)

with torch.no_grad():
    model_tie.fc_out.weight.zero_()
    model_tie.fc_out.bias.fill_(-20.0)
    # Token 5 dan token 12 diberi skor kembar tertinggi
    model_tie.fc_out.bias[5] = 30.0
    model_tie.fc_out.bias[12] = 30.0

f_tie = torch.randn(feature_dim, device=device)
greedy_tie_out = model_tie.generate_greedy(f_tie, max_len=5)
beam_tie_out = model_tie.generate_beam(f_tie, beam_size=1, alpha=0.0, max_len=5)

print(f"Uji 1 - Greedy Tie Out : {greedy_tie_out}")
print(f"Uji 1 - Beam Tie Out   : {beam_tie_out}")

assert greedy_tie_out == beam_tie_out, (
    f"Mismatch pada skor seri! Greedy: {greedy_tie_out}, Beam: {beam_tie_out}"
)
assert greedy_tie_out[0] == 5, (
    f"Token ID terkecil (5) harus terpilih, tetapi didapat {greedy_tie_out[0]}"
)
print("✅ Uji 1: Penanganan skor seri deterministik lolos — token ID terkecil diprioritaskan.")

# -------------------------------------------------------------------------
# Uji 2: Ekuivalensi Menyeluruh Greedy vs Beam (K=1, alpha=0.0)
# -------------------------------------------------------------------------
model_rand = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)
for test_seed in [42, 100, 2026]:
    torch.manual_seed(test_seed)
    f_rand = torch.randn(feature_dim, device=device)
    out_g = model_rand.generate_greedy(f_rand, max_len=15)
    out_b = model_rand.generate_beam(f_rand, beam_size=1, alpha=0.0, max_len=15)
    assert out_g == out_b, f"Seed {test_seed} tidak cocok antara greedy dan beam_size=1!"

print("✅ Uji 2: Ekuivalensi identik antara generate_greedy dan generate_beam(K=1, alpha=0) terverifikasi.")

# -------------------------------------------------------------------------
# Uji 3: Batas Panjang Terkontrol (END dimask ke -inf secara mutlak)
# -------------------------------------------------------------------------
model_no_end = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)
with torch.no_grad():
    model_no_end.fc_out.weight.zero_()
    # Seluruh token dimatikan ke -inf
    model_no_end.fc_out.bias.fill_(-float("inf"))
    # Hanya token 4 yang finite
    model_no_end.fc_out.bias[4] = 10.0

out_no_end = model_no_end.generate_beam(f_tie, beam_size=3, max_len=12)
assert out_no_end == [4] * 12, f"Diharapkan {[4] * 12}, didapat {out_no_end}"
assert len(out_no_end) == 12, f"Panjang harus tepat 12, didapat {len(out_no_end)}"
print("✅ Uji 3: Batas langkah max_len=12 dan penolakan skor non-finite lolos.")

# -------------------------------------------------------------------------
# Uji 4: Terminasi Dini END dan Pengecualian dari Hasil
# -------------------------------------------------------------------------
model_end = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)
with torch.no_grad():
    model_end.fc_out.weight.zero_()
    model_end.fc_out.bias.fill_(-10.0)
    # END (2) mendominasi mutlak
    model_end.fc_out.bias[2] = 50.0

out_end = model_end.generate_beam(f_tie, beam_size=3, max_len=10)
assert out_end == [], f"Diharapkan list kosong [], didapat {out_end}"
print("✅ Uji 4: Kasus END langkah pertama berhenti seketika dan menghasilkan [] bersih.")

# -------------------------------------------------------------------------
# Uji 5: Pemulihan Mode Model dan Proteksi Gradien
# -------------------------------------------------------------------------
model_rand.train()
_ = model_rand.generate_beam(f_tie, beam_size=3, max_len=5)
assert model_rand.training is True, "Mode train gagal dipulihkan!"

model_rand.eval()
_ = model_rand.generate_beam(f_tie, beam_size=3, max_len=5)
assert model_rand.training is False, "Mode eval gagal dipulihkan!"

f_grad = torch.randn(feature_dim, device=device, requires_grad=True)
_ = model_rand.generate_beam(f_grad, beam_size=3, max_len=5)
assert f_grad.grad is None, "Inferensi beam search tidak boleh menghasilkan gradien pada fitur!"
print("✅ Uji 5: Pemulihan mode train/eval dan isolasi gradien terverifikasi aman.")

print("\n--- SELURUH PENGUJIAN UNIT GENERATE_BEAM SELESAI DIRANCANG ---")

--- MEMULAI PENGUJIAN UNIT GENERATE_BEAM ---
Uji 1 - Greedy Tie Out : [5, 5, 5, 5, 5]
Uji 1 - Beam Tie Out   : [5, 5, 5, 5, 5]
✅ Uji 1: Penanganan skor seri deterministik lolos — token ID terkecil diprioritaskan.
✅ Uji 2: Ekuivalensi identik antara generate_greedy dan generate_beam(K=1, alpha=0) terverifikasi.
✅ Uji 3: Batas langkah max_len=12 dan penolakan skor non-finite lolos.
✅ Uji 4: Kasus END langkah pertama berhenti seketika dan menghasilkan [] bersih.
✅ Uji 5: Pemulihan mode train/eval dan isolasi gradien terverifikasi aman.

--- SELURUH PENGUJIAN UNIT GENERATE_BEAM SELESAI DIRANCANG ---


In [9]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import CaptionDecoder

print("=== MEMULAI PENGUJIAN PELENGKAP GENERATE_BEAM ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================================
# Cakupan 1: Model Uji Terkontrol dengan Probabilitas Bersyarat Valid
# Greedy terjebak di optimum lokal, Beam (K=3) menemukan skor global lebih tinggi
# =========================================================================
# Kosakata kecil V=6: {0: PAD, 1: START, 2: END, 3: A, 4: B, 5: C}
V_small = 6
alpha_test = 0.7

# Matriks distribusi probabilitas bersyarat P(next_token | current_token)
# Setiap baris merepresentasikan distribusi diskret valid yang jumlahnya tepat 1.0
# Kolom:   PAD(0) START(1) END(2)   A(3)   B(4)   C(5)
trans_probs = torch.zeros(V_small, V_small)

# Dari START (1): A=0.55, B=0.45 (PAD, START, END = 0.0) -> Jumlah = 1.0
trans_probs[1, 3] = 0.55
trans_probs[1, 4] = 0.45

# Dari A (3): C=0.60, END=0.40 -> Jumlah = 1.0
trans_probs[3, 5] = 0.60
trans_probs[3, 2] = 0.40

# Dari B (4): END=0.95, C=0.05 -> Jumlah = 1.0
trans_probs[4, 2] = 0.95
trans_probs[4, 5] = 0.05

# Dari C (5): END=0.60, C=0.40 -> Jumlah = 1.0
trans_probs[5, 2] = 0.60
trans_probs[5, 5] = 0.40

# Ubah probabilitas menjadi logit (log-probabilitas)
trans_logits = torch.full((V_small, V_small), -100.0)
nonzero_mask = trans_probs > 0
trans_logits[nonzero_mask] = torch.log(trans_probs[nonzero_mask])

# Buat decoder uji dengan dimensi tereduksi
model_markov = CaptionDecoder(
    vocab_size=V_small,
    feature_dim=2048,
    embed_dim=V_small,
    hidden_size=V_small,
    dropout_p=0.0
).to(device)

# Modul perantara transisi agar satu langkah waktu menghasilkan logit sesuai trans_logits[token_aktif]
class MarkovLSTM(nn.Module):
    def __init__(self, logits_matrix):
        super().__init__()
        # Transpose agar baris fc_out mewakili token target sesuai one-hot input
        self.logits_matrix = logits_matrix.to(device)
    def forward(self, emb, states):
        # emb: [1, 1, V_small] (one-hot dari token aktif)
        h, c = states
        # Proyeksikan langsung vektor one-hot ke baris logit yang bersesuaian
        out_logits = torch.matmul(emb, self.logits_matrix)
        return out_logits, (h + 1.0, c + 1.0)

with torch.no_grad():
    model_markov.embedding.weight.copy_(torch.eye(V_small, device=device))
    model_markov.lstm = MarkovLSTM(trans_logits)
    model_markov.fc_out.weight.copy_(torch.eye(V_small, device=device))
    model_markov.fc_out.bias.zero_()

f_dummy = torch.randn(2048, device=device)

# Eksekusi inferensi
greedy_res = model_markov.generate_greedy(f_dummy, max_len=10)
beam_res = model_markov.generate_beam(f_dummy, beam_size=3, max_len=10, alpha=alpha_test)

# --- Perhitungan Manual Terverifikasi ---
# 1. Jalur Greedy:
#    Langkah 1: START -> pilih A (P=0.55 > B=0.45)
#    Langkah 2: Dari A -> pilih C (P=0.60 > END=0.40)
#    Langkah 3: Dari C -> pilih END (P=0.60 > C=0.40) -> berhenti
#    Token yang dihasilkan: [3, 5] (A, C)
#    L_greedy = 3 (A, C, END dihitung; START tidak dihitung)
cum_log_greedy = math.log(0.55) + math.log(0.60) + math.log(0.60) # log(0.198) = -1.619488
lp_greedy = ((5.0 + 3) / 6.0) ** alpha_test                     # (8/6)**0.7 = 1.223038
expected_score_greedy = cum_log_greedy / lp_greedy               # ≈ -1.324152

# 2. Jalur Alternatif Beam Search (via B):
#    Langkah 1: Beam mempertahankan A (0.55) dan B (0.45)
#    Langkah 2: Dari B -> pilih END (P=0.95) -> kandidat selesai
#    Token yang dihasilkan: [4] (B)
#    L_beam = 2 (B, END dihitung; START tidak dihitung)
cum_log_beam = math.log(0.45) + math.log(0.95)                   # log(0.4275) = -0.849805
lp_beam = ((5.0 + 2) / 6.0) ** alpha_test                       # (7/6)**0.7 = 1.113618
expected_score_beam = cum_log_beam / lp_beam                     # ≈ -0.763103

print(f"Hasil Token Greedy   : {greedy_res} (Skor ternormalisasi manual: {expected_score_greedy:.4f})")
print(f"Hasil Token Beam (3) : {beam_res} (Skor ternormalisasi manual: {expected_score_beam:.4f})")

assert greedy_res == [3, 5], f"Greedy salah jalur! Didapat: {greedy_res}"
assert beam_res == [4], f"Beam search gagal menemukan jalur global terbaik! Didapat: {beam_res}"
assert expected_score_beam > expected_score_greedy, "Skor beam harus lebih tinggi dari skor greedy!"
print("✅ Cakupan 1 Lolos: Greedy memilih jalur lokal [3, 5], sedangkan Beam Search (3) menemukan [4] dengan skor akhir lebih tinggi.")

# =========================================================================
# Cakupan 2: Verifikasi Cabang Selesai Tidak Diproses Ulang oleh LSTM
# =========================================================================
# Gunakan forward hook pada lstm untuk mencatat setiap pemanggilan
lstm_inputs_token = []
def lstm_hook(module, inp, outp):
    # inp[0] adalah tensor embedding [1, 1, embed_dim]
    emb_vec = inp[0].squeeze()
    token_id = torch.argmax(emb_vec).item()
    lstm_inputs_token.append(token_id)

hook_handle = model_markov.lstm.register_forward_hook(lstm_hook)

# Kasus 2a: Hanya token END (2) yang bernilai logit finite sejak langkah awal
model_instant_end = CaptionDecoder(vocab_size=3000, feature_dim=2048).to(device)
with torch.no_grad():
    model_instant_end.fc_out.weight.zero_()
    model_instant_end.fc_out.bias.fill_(-float("inf"))
    model_instant_end.fc_out.bias[2] = 10.0  # Hanya END yang valid

lstm_calls_end_only = 0
def count_hook(module, inp, outp):
    global lstm_calls_end_only
    lstm_calls_end_only += 1

h_count = model_instant_end.lstm.register_forward_hook(count_hook)
out_instant = model_instant_end.generate_beam(f_dummy, beam_size=3, max_len=10)
h_count.remove()

# Terbukti loop berhenti tepat pada langkah pertama karena seluruh kandidat selesai
assert out_instant == [], "Output harus berupa []"
assert lstm_calls_end_only == 1, (
    f"LSTM seharusnya hanya dipanggil tepat 1 kali pada langkah ke-1, tetapi tercatat {lstm_calls_end_only} kali!"
)

# Kasus 2b: Percabangan aktif di mana cabang END tidak diekspansi lagi di langkah ke-2
lstm_inputs_token.clear()
_ = model_markov.generate_beam(f_dummy, beam_size=3, max_len=5, alpha=alpha_test)
hook_handle.remove()

# Pada langkah 1, START (1) diekspansi menghasilkan cabang A (3) dan B (4).
# Pada langkah 2, A dan B diekspansi. Cabang B langsung memprediksi END dan berstatus is_done=True.
# Pada langkah 3, hanya kandidat yang belum selesai yang diekspansi; token END (2) TIDAK PERNAH masuk LSTM!
assert 2 not in lstm_inputs_token, (
    f"Token END (2) terdeteksi diproses ulang oleh LSTM! Riwayat pemanggilan token: {lstm_inputs_token}"
)
print(f"✅ Cakupan 2 Lolos: Terbukti bahwa saat seluruh kandidat selesai loop berhenti di langkah 1 (total panggil: {lstm_calls_end_only}), "
      f"serta cabang END tidak pernah diproses kembali oleh LSTM (riwayat token masuk LSTM: {lstm_inputs_token}).")

# =========================================================================
# Cakupan 3: Uji Masking PAD dan START dengan Berbagai Token Isi Tersedia
# =========================================================================
model_mask = CaptionDecoder(vocab_size=3000, feature_dim=2048).to(device)
with torch.no_grad():
    model_mask.fc_out.weight.zero_()
    model_mask.fc_out.bias.fill_(-50.0)
    # PAD (0) dan START (1) memiliki logit paling tinggi
    model_mask.fc_out.bias[0] = 100.0  # PAD
    model_mask.fc_out.bias[1] = 90.0   # START
    # Beberapa token isi tetap tersedia dengan logit finite
    model_mask.fc_out.bias[6] = 20.0
    model_mask.fc_out.bias[7] = 15.0
    model_mask.fc_out.bias[8] = 10.0

out_mask = model_mask.generate_beam(f_dummy, beam_size=3, max_len=8)
assert 0 not in out_mask, "Token PAD (0) tidak boleh terpilih!"
assert 1 not in out_mask, "Token START (1) tidak boleh terpilih!"
assert 2 not in out_mask, "Token END (2) tidak boleh ada pada list hasil!"
assert set(out_mask).issubset({6, 7, 8}), f"Token yang dihasilkan harus berasal dari token isi valid, didapat: {out_mask}"
print(f"✅ Cakupan 3 Lolos: Masking PAD (0) dan START (1) bekerja mutlak meskipun memiliki bias awal tertinggi (hasil: {out_mask[:4]}...).")

# =========================================================================
# Cakupan 4: Uji Input [2048], [1, 2048], dan Penolakan Bentuk Tidak Valid
# =========================================================================
f_1d = torch.randn(2048, device=device)
f_2d = torch.randn(1, 2048, device=device)

# Input valid harus berjalan tanpa galat
res_1d = model_mask.generate_beam(f_1d, beam_size=2, max_len=5)
res_2d = model_mask.generate_beam(f_2d, beam_size=2, max_len=5)
assert isinstance(res_1d, list) and isinstance(res_2d, list)

# Penolakan bentuk salah
invalid_shapes = [(1024,), (2, 2048), (1, 1, 2048)]
rejection_count = 0
for bad_shape in invalid_shapes:
    try:
        bad_f = torch.randn(*bad_shape, device=device)
        model_mask.generate_beam(bad_f, beam_size=2, max_len=5)
    except ValueError as e:
        rejection_count += 1

assert rejection_count == len(invalid_shapes), "Seluruh bentuk invalid wajib memicu ValueError!"
print("✅ Cakupan 4 Lolos: Input 1D [2048] dan 2D [1, 2048] diterima; tensor [1024], [2, 2048], dan [1, 1, 2048] ditolak via ValueError.")

# =========================================================================
# Cakupan 5: Integritas Gradien Parameter Eksisting Sebelum & Sesudah Inferensi
# =========================================================================
model_grad = CaptionDecoder(vocab_size=3000, feature_dim=2048).to(device)
model_grad.train()

# Simulasikan langkah backward sebelum inferensi untuk menghasilkan gradien parameter
dummy_feats = torch.randn(2, 2048, device=device)
dummy_caps_in = torch.randint(0, 3000, (2, 29), device=device)
dummy_logits = model_grad(dummy_feats, dummy_caps_in)
dummy_loss = dummy_logits.sum()
dummy_loss.backward()

# Catat snapshot gradien sebelum inferensi
grad_init_h_before = model_grad.init_h.weight.grad.clone()
grad_fc_out_before = model_grad.fc_out.weight.grad.clone()
assert grad_init_h_before is not None and grad_fc_out_before is not None

# Jalankan inferensi beam search dengan tensor fitur ber-gradien
feat_with_grad = torch.randn(2048, device=device, requires_grad=True)
_ = model_grad.generate_beam(feat_with_grad, beam_size=3, max_len=5)

# Periksa bahwa tensor masukan inferensi tidak memiliki gradien baru
assert feat_with_grad.grad is None, "Tensor masukan inferensi tidak boleh menghasilkan gradien!"

# Periksa bahwa gradien parameter yang sudah ada tetap identik tanpa modifikasi/penghapusan
assert torch.equal(model_grad.init_h.weight.grad, grad_init_h_before), "Gradien init_h berubah setelah inferensi!"
assert torch.equal(model_grad.fc_out.weight.grad, grad_fc_out_before), "Gradien fc_out berubah setelah inferensi!"
print("✅ Cakupan 5 Lolos: Gradien parameter eksisting tetap identik (torch.equal) dan fitur inferensi bebas gradien.")

print("\n=== SELURUH PENGUJIAN PELENGKAP BERHASIL DAN TERVERIFIKASI LENGKAP ===")

=== MEMULAI PENGUJIAN PELENGKAP GENERATE_BEAM ===
Hasil Token Greedy   : [3, 5] (Skor ternormalisasi manual: -1.3241)
Hasil Token Beam (3) : [4] (Skor ternormalisasi manual: -0.7629)
✅ Cakupan 1 Lolos: Greedy memilih jalur lokal [3, 5], sedangkan Beam Search (3) menemukan [4] dengan skor akhir lebih tinggi.
✅ Cakupan 2 Lolos: Terbukti bahwa saat seluruh kandidat selesai loop berhenti di langkah 1 (total panggil: 1), serta cabang END tidak pernah diproses kembali oleh LSTM (riwayat token masuk LSTM: [1, 3, 4, 5]).
✅ Cakupan 3 Lolos: Masking PAD (0) dan START (1) bekerja mutlak meskipun memiliki bias awal tertinggi (hasil: [6, 6, 6, 6]...).
✅ Cakupan 4 Lolos: Input 1D [2048] dan 2D [1, 2048] diterima; tensor [1024], [2, 2048], dan [1, 1, 2048] ditolak via ValueError.
✅ Cakupan 5 Lolos: Gradien parameter eksisting tetap identik (torch.equal) dan fitur inferensi bebas gradien.

=== SELURUH PENGUJIAN PELENGKAP BERHASIL DAN TERVERIFIKASI LENGKAP ===
